In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv("../data/job_descriptions.csv")

In [ ]:
df.head()

,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Contact,Job Title,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile
0,1089843540111562,5 to 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,54.2361,-4.5481,Intern,26801,...,001-381-930-7517x737,Digital Marketing Specialist,Social Media Manager,Snagajob,Social Media Managers oversee an organizations...,"{'Flexible Spending Accounts (FSAs), Relocatio...","Social media platforms (e.g., Facebook, Twitte...","Manage and grow social media accounts, create ...",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversifie..."
1,398454096642776,2 to 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,38.9697,59.5563,Intern,100340,...,461-509-4216,Web Developer,Frontend Web Developer,Idealist,Frontend Web Developers design and implement u...,"{'Health Insurance, Retirement Plans, Paid Tim...","HTML, CSS, JavaScript Frontend frameworks (e.g...","Design and code user interfaces for websites, ...",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Com..."
2,481640072963533,0 to 12 Years,PhD,$61K-$104K,Macao,"Macao SAR, China",22.1987,113.5439,Temporary,84525,...,9687619505,Operations Manager,Quality Control Manager,Jobs2Careers,Quality Control Managers establish and enforce...,"{'Legal Assistance, Bonuses and Incentive Prog...",Quality control processes and methodologies St...,Establish and enforce quality control standard...,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: P..."
3,688192671473044,4 to 11 Years,PhD,$65K-$91K,Porto-Novo,Benin,9.3077,2.3158,Full-Time,129896,...,+1-820-643-5431x47576,Network Engineer,Wireless Network Engineer,FlexJobs,"Wireless Network Engineers design, implement, ...","{'Transportation Benefits, Professional Develo...",Wireless network design and architecture Wi-Fi...,"Design, configure, and optimize wireless netwo...",Hess,"{""Sector"":""Energy"",""Industry"":""Mining, Crude-O..."
4,117057806156508,1 to 12 Years,MBA,$64K-$87K,Santiago,Chile,-35.6751,-71.5429,Intern,53944,...,343.975.4702x9340,Event Manager,Conference Manager,Jobs2Careers,A Conference Manager coordinates and manages c...,"{'Flexible Spending Accounts (FSAs), Relocatio...",Event planning Conference logistics Budget man...,Specialize in conference and convention planni...,Cairn Energy,"{""Sector"":""Energy"",""Industry"":""Energy - Oil & ..."


In [ ]:
df.shape

(1615940, 23)

In [ ]:
df.columns

Index(['Job Id', 'Experience', 'Qualifications', 'Salary Range', 'location',
       'Country', 'latitude', 'longitude', 'Work Type', 'Company Size',
       'Job Posting Date', 'Preference', 'Contact Person', 'Contact',
       'Job Title', 'Role', 'Job Portal', 'Job Description', 'Benefits',
       'skills', 'Responsibilities', 'Company', 'Company Profile'],
      dtype='object')

In [ ]:
df.isnull().sum()

Job Id                 0
Experience             0
Qualifications         0
Salary Range           0
location               0
Country                0
latitude               0
longitude              0
Work Type              0
Company Size           0
Job Posting Date       0
Preference             0
Contact Person         0
Contact                0
Job Title              0
Role                   0
Job Portal             0
Job Description        0
Benefits               0
skills                 0
Responsibilities       0
Company                0
Company Profile     5478
dtype: int64

In [ ]:
df["combined_text"]=(
    df["Job Title"].astype(str) + " " +
    df["Role"].astype(str) + " " +
    df["Job Description"].astype(str) + " " +
    df["skills"].astype(str) + " " +
    df["Qualifications"].astype(str) + " " +
    df["Responsibilities"].astype(str) 

)

In [ ]:
df[["Job Title", "combined_text"]].head()

,Job Title,combined_text
0,Digital Marketing Specialist,Digital Marketing Specialist Social Media Mana...
1,Web Developer,Web Developer Frontend Web Developer Frontend ...
2,Operations Manager,Operations Manager Quality Control Manager Qua...
3,Network Engineer,Network Engineer Wireless Network Engineer Wir...
4,Event Manager,Event Manager Conference Manager A Conference ...


In [ ]:
df=df[["Job Title", "combined_text"]]

In [ ]:
df.head()

,Job Title,combined_text
0,Digital Marketing Specialist,Digital Marketing Specialist Social Media Mana...
1,Web Developer,Web Developer Frontend Web Developer Frontend ...
2,Operations Manager,Operations Manager Quality Control Manager Qua...
3,Network Engineer,Network Engineer Wireless Network Engineer Wir...
4,Event Manager,Event Manager Conference Manager A Conference ...


In [ ]:
df=df.sample(50000, random_state=42)

In [ ]:
df.shape

(50000, 2)

In [ ]:
import re
import nltk

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\prateek.m\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
from nltk.corpus import stopwords

In [ ]:
stop_words=set(stopwords.words("english"))

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)   # remove numbers & symbols
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

In [ ]:
df["cleaned_text"] = df["combined_text"].apply(clean_text)

In [ ]:
df[["Job Title", "cleaned_text"]].head()

,Job Title,cleaned_text
108318,Procurement Manager,procurement manager supplier diversity manager...
7787,Architectural Designer,architectural designer architectural drafter a...
1237496,Art Teacher,art teacher art education coordinator art educ...
55757,Environmental Consultant,environmental consultant environmental impact ...
818970,Art Teacher,art teacher art education coordinator art educ...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf=TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

In [ ]:
tfidf_matrix = tfidf.fit_transform(df["cleaned_text"])

In [ ]:
tfidf_matrix.shape

(50000, 5000)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def recommend_jobs(job_index, top_n=5):
    similarity_scores = cosine_similarity(tfidf_matrix[job_index], tfidf_matrix)
    scores = similarity_scores.flatten()
    
    similar_indices = scores.argsort()[::-1][1:top_n+1]
    
    return df.iloc[similar_indices][["Job Title"]]

In [ ]:
recommend_jobs(0)

,Job Title
740336,Procurement Manager
1369017,Procurement Manager
1142513,Procurement Manager
1028913,Procurement Manager
955993,Procurement Manager


In [ ]:
recommend_jobs(100)

,Job Title
91577,UX/UI Designer
966172,UX/UI Designer
962151,UX/UI Designer
933652,UX/UI Designer
29139,UX/UI Designer


In [ ]:
def recommend_from_text(user_text, top_n=5):
    cleaned = clean_text(user_text)
    vector = tfidf.transform([cleaned])
    
    similarity = cosine_similarity(vector, tfidf_matrix)
    scores = similarity.flatten()
    
    top_indices = scores.argsort()[::-1][:top_n]
    
    return df.iloc[top_indices][["Job Title"]]

In [ ]:
recommend_from_text("python machine learning data analysis")

,Job Title
1037655,Data Scientist
675761,Data Scientist
1466038,Data Scientist
110525,Data Scientist
222130,Data Scientist


In [ ]:
recommend_from_text("network security cloud infrastructure")

,Job Title
1431414,Network Technician
1231568,Network Technician
1401000,Network Technician
1350617,Network Technician
697740,Network Technician


In [ ]:
import pickle

In [ ]:
pickle.dump(tfidf, open("../src/model/tfidf_vectorizer.pkl", "wb"))
pickle.dump(tfidf_matrix, open("../src/model/tfidf_matrix.pkl", "wb"))
pickle.dump(df, open("../src/model/jobs_dataframe.pkl", "wb"))

In [36]:
df.sample(1000).to_csv("data/sample_jobs.csv", index=False)

OSError: Cannot save file into a non-existent directory: 'data'

In [37]:
df.sample(1000).to_csv("../data/sample_jobs.csv", index=False)

In [38]:
import pickle

pickle.dump(tfidf, open("../src/model/tfidf_vectorizer.pkl", "wb"))
pickle.dump(tfidf_matrix, open("../src/model/tfidf_matrix.pkl", "wb"))
pickle.dump(df, open("../src/model/jobs_dataframe.pkl", "wb"))